In [1]:
import polars as pl


In [44]:
df = pl.read_parquet('data/results.parquet')

In [45]:
df = df.filter(pl.col('vorlagenId') == 6810)

In [46]:
# Group smaller cantons together based on geographical proximity

mapping = {
    'Zürich': 'Zürich',
    'Bern': 'Bern',
    'Luzern': 'Zentralschweiz',
    'Uri': 'Zentralschweiz',
    'Schwyz': 'Zentralschweiz',
    'Obwalden': 'Zentralschweiz',
    'Nidwalden': 'Zentralschweiz',
    'Zug': 'Zentralschweiz',
    'Aargau': 'Aargau',
    'Thurgau': 'Ostschweiz',
    'St. Gallen': 'Ostschweiz',
    'Appenzell Ausserrhoden': 'Ostschweiz',
    'Appenzell Innerrhoden': 'Ostschweiz',
    'Schaffhausen': 'Ostschweiz',
    'Glarus': 'Ostschweiz',
    'Graubünden': 'Graubünden',
    'Jura': 'Jura',
    'Solothurn': 'Solothurn',
    'Basel-Stadt': 'Basel',
    'Basel-Landschaft': 'Basel',
    'Ticino': 'Ticino',
    'Vaud': 'Vaud',
    'Fribourg': 'Fribourg',
    'Neuchâtel': 'Neuchâtel',
    'Genève': 'Genève',
    'Valais': 'Valais',
}


In [47]:
df

geoLevelnummer,geoLevelname,geoLevelParentnummer,vorlagenId,kanton,gebietAusgezaehlt,jaStimmenInProzent,jaStimmenAbsolut,neinStimmenAbsolut,stimmbeteiligungInProzent,eingelegteStimmzettel,anzahlStimmberechtigte,gueltigeStimmen
i64,str,str,i64,str,bool,f64,i64,i64,f64,i64,i64,i64
1,"""Aeugst am Albis""","""101""",6810,"""Zürich""",true,15.953307,123,648,56.423358,773,1370,771
2,"""Affoltern am Albis""","""101""",6810,"""Zürich""",true,18.450065,569,2515,42.695856,3101,7263,3084
3,"""Bonstetten""","""101""",6810,"""Zürich""",true,19.826707,389,1573,53.913281,1977,3667,1962
4,"""Hausen am Albis""","""101""",6810,"""Zürich""",true,22.295806,303,1056,52.107862,1372,2633,1359
5,"""Hedingen""","""101""",6810,"""Zürich""",true,20.697168,285,1092,53.667954,1390,2590,1377
…,…,…,…,…,…,…,…,…,…,…,…,…
6808,"""Clos du Doubs""","""2603""",6810,"""Jura""",true,26.969697,89,241,29.462738,340,1154,330
6809,"""Haute-Ajoie""","""2603""",6810,"""Jura""",true,26.807229,89,243,32.843137,335,1020,332
6810,"""La Baroche""","""2603""",6810,"""Jura""",true,25.0,75,225,30.33367,300,989,300


In [48]:
df = df.with_columns(
    pl.col('kanton').replace_strict(mapping, default='???').alias('region')
)

df.group_by('region').len().sort('len').head(20)

region,len
str,u32
"""Neuchâtel""",24
"""Genève""",46
"""Jura""",50
"""Basel""",90
"""Graubünden""",100
…,…
"""Zürich""",161
"""Aargau""",198
"""Ostschweiz""",212


In [49]:
df = df.with_columns(
    enthaltung=pl.col('anzahlStimmberechtigte') - pl.col('jaStimmenAbsolut') - pl.col('neinStimmenAbsolut')
)

In [50]:
df_nrw = pl.read_parquet('data/nrw.parquet')

In [51]:
df_nrw

gemeinde_nummer,FDP,SP,SVP,EVP,PdA/Sol.,FGA,GRÜNE,SD,EDU,GLP,Mitte,Übrige,CSP,Lega,MCR,nichtwahler
i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
1,104,111,279,31,2,2,64,3,19,118,47,35,0,0,0,568
2,376,516,1056,197,2,10,265,6,38,409,277,126,0,0,0,4001
3,280,355,562,75,5,10,158,2,24,315,143,70,0,0,0,1640
4,143,245,471,40,3,5,138,0,18,167,96,53,0,0,0,1240
5,221,237,476,45,3,1,112,1,30,243,99,54,0,0,0,1049
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
19190,237,430,509,50,5,0,346,0,42,254,155,63,0,0,0,10346
19200,105,150,234,29,0,0,172,0,22,101,116,26,0,0,0,4001
19220,629,918,821,64,190,0,718,0,78,361,226,100,0,0,0,20054


In [52]:
df_joined = df.select(
        ['geoLevelnummer', 'region', 'jaStimmenAbsolut', 'neinStimmenAbsolut', 'enthaltung']
    ).join(df_nrw, left_on='geoLevelnummer', right_on='gemeinde_nummer', how='left').drop_nulls().drop('geoLevelnummer')

In [53]:
df_joined

region,jaStimmenAbsolut,neinStimmenAbsolut,enthaltung,FDP,SP,SVP,EVP,PdA/Sol.,FGA,GRÜNE,SD,EDU,GLP,Mitte,Übrige,CSP,Lega,MCR,nichtwahler
str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
"""Zürich""",123,648,599,104,111,279,31,2,2,64,3,19,118,47,35,0,0,0,568
"""Zürich""",569,2515,4179,376,516,1056,197,2,10,265,6,38,409,277,126,0,0,0,4001
"""Zürich""",389,1573,1705,280,355,562,75,5,10,158,2,24,315,143,70,0,0,0,1640
"""Zürich""",303,1056,1274,143,245,471,40,3,5,138,0,18,167,96,53,0,0,0,1240
"""Zürich""",285,1092,1213,221,237,476,45,3,1,112,1,30,243,99,54,0,0,0,1049
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Jura""",51,234,669,49,94,90,4,0,0,18,0,0,20,156,5,0,0,0,532
"""Jura""",89,241,824,23,124,110,3,0,0,48,0,0,8,175,15,0,0,0,628
"""Jura""",89,243,688,40,95,91,1,0,0,38,0,0,17,189,4,0,0,0,522


In [54]:
from rpy2.robjects import pandas2ri, numpy2ri, default_converter

In [55]:
import rpy2.robjects as ro
from rpy2.robjects.packages import importr
from rpy2.robjects.vectors import FloatVector

In [56]:
base = importr('base')
utils = importr('utils')

# Load the lphom package
lphom = importr('lphom')

In [57]:
df_joined

region,jaStimmenAbsolut,neinStimmenAbsolut,enthaltung,FDP,SP,SVP,EVP,PdA/Sol.,FGA,GRÜNE,SD,EDU,GLP,Mitte,Übrige,CSP,Lega,MCR,nichtwahler
str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
"""Zürich""",123,648,599,104,111,279,31,2,2,64,3,19,118,47,35,0,0,0,568
"""Zürich""",569,2515,4179,376,516,1056,197,2,10,265,6,38,409,277,126,0,0,0,4001
"""Zürich""",389,1573,1705,280,355,562,75,5,10,158,2,24,315,143,70,0,0,0,1640
"""Zürich""",303,1056,1274,143,245,471,40,3,5,138,0,18,167,96,53,0,0,0,1240
"""Zürich""",285,1092,1213,221,237,476,45,3,1,112,1,30,243,99,54,0,0,0,1049
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Jura""",51,234,669,49,94,90,4,0,0,18,0,0,20,156,5,0,0,0,532
"""Jura""",89,241,824,23,124,110,3,0,0,48,0,0,8,175,15,0,0,0,628
"""Jura""",89,243,688,40,95,91,1,0,0,38,0,0,17,189,4,0,0,0,522


In [58]:
PARTEIN = [
    "FDP", "SP", "SVP", "EVP", "PdA/Sol.", "FGA", "GRÜNE", "SD", "EDU", "GLP", "Mitte", "Übrige", "CSP", "Lega", "MCR", "nichtwahler",
]

In [59]:
def estimate_voter_behavior(df_joined):
    vote1 = df_joined.select(PARTEIN)
    vote2 = df_joined.select([
        "jaStimmenAbsolut", "neinStimmenAbsolut", "enthaltung"
    ])
    with (ro.default_converter + pandas2ri.converter).context():
        v1 = ro.conversion.py2rpy(vote1.to_pandas())
        v2 = ro.conversion.py2rpy(vote2.to_pandas())
        res = lphom.lphom(v1, v2)
    return dict(zip(res.names(), res.values()))

In [60]:
import numpy as np

In [61]:
def res_to_df(array):

    df = pl.from_numpy(array, schema=['ja', 'nein', 'enthaltung', 'exit'])
    return df.with_columns(
        partei=pl.Series(PARTEIN + ['neuwähler'])
    ).select(['partei', 'ja', 'nein', 'enthaltung', 'exit'])

In [62]:
total = None

for kanton in df_joined['region'].unique():
    print(f"## {kanton}")
    res = estimate_voter_behavior(df_joined.filter(pl.col('region') == kanton))
    if total is None:
        total = res['VTM.complete.votes']
    else:
        total += res['VTM.complete.votes']


R callback write-console: *********************WARNING*********************
You are in a "raw" scenario.
The sums (by row) of origin and destination data differ. It is, for at least a unit, the total number of electors in both elections is not the same. 
To guarantee the matching: A new category of census entries (NET_ENTRIES) has been included in the origin election and a new category of census exits (NET_EXITS) has been also included in the destination election.
 Their aggregate importances, measured in percentage of the total census, are:
     %NET_ENTRIES = 2.6577%
     %NET_EXITS = 0.231%
If NET_ENTRIES and/or NET_EXITS are really small, less than 1% in all units, their corresponding results will not be displayed in the main output, VTM. They are anyway always included in VTM.complete. 
*************************************************
  
R callback write-console: *********************WARNING*********************
You are in a "raw" scenario.
The sums (by row) of origin and destina

## Basel
## Graubünden
## Zentralschweiz


R callback write-console: *********************WARNING*********************
You are in a "raw" scenario.
The sums (by row) of origin and destination data differ. It is, for at least a unit, the total number of electors in both elections is not the same. 
To guarantee the matching: A new category of census entries (NET_ENTRIES) has been included in the origin election and a new category of census exits (NET_EXITS) has been also included in the destination election.
 Their aggregate importances, measured in percentage of the total census, are:
     %NET_ENTRIES = 2.5675%
     %NET_EXITS = 0.1293%
If NET_ENTRIES and/or NET_EXITS are really small, less than 1% in all units, their corresponding results will not be displayed in the main output, VTM. They are anyway always included in VTM.complete. 
*************************************************
  
R callback write-console: *********************WARNING*********************
You are in a "raw" scenario.
The sums (by row) of origin and destin

## Fribourg
## Neuchâtel
## Ostschweiz


R callback write-console: *********************WARNING*********************
You are in a "raw" scenario.
The sums (by row) of origin and destination data differ. It is, for at least a unit, the total number of electors in both elections is not the same. 
To guarantee the matching: A new category of census entries (NET_ENTRIES) has been included in the origin election and a new category of census exits (NET_EXITS) has been also included in the destination election.
 Their aggregate importances, measured in percentage of the total census, are:
     %NET_ENTRIES = 2.3749%
     %NET_EXITS = 0.0312%
If NET_ENTRIES and/or NET_EXITS are really small, less than 1% in all units, their corresponding results will not be displayed in the main output, VTM. They are anyway always included in VTM.complete. 
*************************************************
  
R callback write-console: *********************WARNING*********************
You are in a "raw" scenario.
The sums (by row) of origin and destin

...

## Valais
## Bern


R callback write-console: *********************WARNING*********************
You are in a "raw" scenario.
The sums (by row) of origin and destination data differ. It is, for at least a unit, the total number of electors in both elections is not the same. 
To guarantee the matching: A new category of census entries (NET_ENTRIES) has been included in the origin election and a new category of census exits (NET_EXITS) has been also included in the destination election.
 Their aggregate importances, measured in percentage of the total census, are:
     %NET_ENTRIES = 1.8324%
     %NET_EXITS = 0.1541%
If NET_ENTRIES and/or NET_EXITS are really small, less than 1% in all units, their corresponding results will not be displayed in the main output, VTM. They are anyway always included in VTM.complete. 
*************************************************
  


## Aargau


R callback write-console: *********************WARNING*********************
You are in a "raw" scenario.
The sums (by row) of origin and destination data differ. It is, for at least a unit, the total number of electors in both elections is not the same. 
To guarantee the matching: A new category of census entries (NET_ENTRIES) has been included in the origin election and a new category of census exits (NET_EXITS) has been also included in the destination election.
 Their aggregate importances, measured in percentage of the total census, are:
     %NET_ENTRIES = 1.0307%
     %NET_EXITS = 0.3225%
If NET_ENTRIES and/or NET_EXITS are really small, less than 1% in all units, their corresponding results will not be displayed in the main output, VTM. They are anyway always included in VTM.complete. 
*************************************************
  
R callback write-console: *********************WARNING*********************
You are in a "raw" scenario.
The sums (by row) of origin and destin

## Solothurn
## Zürich


In [63]:
res_to_df(total)

partei,ja,nein,enthaltung,exit
str,f64,f64,f64,f64
"""FDP""",9009.557687,335742.215831,14916.048422,877.178059
"""SP""",271396.615191,46973.844785,148486.349093,1133.190931
"""SVP""",196.19088,548294.012648,170280.727213,1702.069259
"""EVP""",429.220745,40118.682285,9390.449926,122.647043
"""PdA/Sol.""",12999.412817,0.0,5124.385898,36.201285
…,…,…,…,…
"""CSP""",62.331128,1697.689313,576.875229,3.104329
"""Lega""",0.0,14350.503257,0.0,90.496743
"""MCR""",0.0,7961.138922,5191.657512,4.203566


In [64]:
import altair as alt

In [ ]:
# Stacked barchart of the results, with one bar per party, showing the distribution of votes (yes, no, abstain, exit)

df_long = res_to_df(total).unpivot(index='partei').with_columns(
    pl.col('value').cast(pl.Int64)
)
df_long

partei,variable,value
str,str,i64
"""FDP""","""ja""",9009
"""SP""","""ja""",271396
"""SVP""","""ja""",196
"""EVP""","""ja""",429
"""PdA/Sol.""","""ja""",12999
…,…,…
"""CSP""","""exit""",3
"""Lega""","""exit""",90
"""MCR""","""exit""",4


In [73]:
list(df_long['partei'].unique())

['EVP',
 'SP',
 'neuwähler',
 'Lega',
 'CSP',
 'MCR',
 'SVP',
 'FGA',
 'GRÜNE',
 'Mitte',
 'FDP',
 'Übrige',
 'nichtwahler',
 'EDU',
 'SD',
 'GLP',
 'PdA/Sol.']

In [74]:
GROSSE_PARTEIEN = ["FDP", "SP", "SVP", "GRÜNE", "Mitte", "GLP", "neuwähler", "EVP" ]

In [75]:
df_long = df_long.filter(pl.col('partei').is_in(GROSSE_PARTEIEN))

In [76]:
# Prepare data for the chart
df_chart = df_long.filter(pl.col('variable').is_in(['ja', 'enthaltung', 'nein']))

# Calculate percentage of 'ja' for each party to sort by
df_ja_percent = df_chart.group_by('partei').agg([
    pl.col('value').sum().alias('total'),
    pl.col('value').filter(pl.col('variable') == 'ja').sum().alias('ja_votes')
]).with_columns(
    (pl.col('ja_votes') / pl.col('total')).alias('ja_percent')
).sort('ja_percent', descending=True)

# Get the sorted party order
party_order = df_ja_percent['partei'].to_list()

# Create the chart with custom color scheme and sorting
chart = alt.Chart(df_chart).mark_bar().encode(
    x=alt.X('sum(value):Q', 
            stack='normalize', 
            title='Anteil der Stimmen'),
    y=alt.Y('partei:N', 
            title='Partei',
            sort=party_order),
    color=alt.Color('variable:N', 
                   title='Stimmverhalten',
                   scale=alt.Scale(
                       domain=['ja', 'enthaltung', 'nein'],
                       range=['#1f77b4', '#7f7f7f', '#ff7f0e']  # blue, grey, orange
                   )),
    tooltip=[
        alt.Tooltip('partei', title='Partei'),
        alt.Tooltip('variable', title='Stimmverhalten'),
    ],
    order=alt.Order('variable_order:O')
).transform_calculate(
    # Create an order field for proper stacking
    variable_order='if(datum.variable == "ja", 1, if(datum.variable == "enthaltung", 2, 3))',
    # Format numbers with thousand separators
    value_formatted='format(datum.value, ",.0f")',
    # Calculate and format percentage
    percentage_formatted='format(datum.value / datum.total_party, ".1%")'
).properties(
    width=600,
    height=400,
    title='Wahlverhalten der Wähler:innen nach Partei (geschätzt)'
)

chart

alt.Chart(...)

In [80]:

chart = alt.Chart(df_chart).mark_bar().encode(
    x=alt.X('sum(value):Q', 
            stack='normalize', 
            title='Anteil der Stimmen'),
    y=alt.Y('partei:N', 
            title='Partei',
            sort=party_order),
    color=alt.Color('variable:N', 
                   title='Stimmverhalten',
                   scale=alt.Scale(
                       domain=['ja', 'enthaltung', 'nein'],
                       range=['#1f77b4', '#7f7f7f', '#ff7f0e']  # blue, grey, orange
                   )),
    tooltip=[
        alt.Tooltip('partei', title='Partei'),
        alt.Tooltip('variable', title='Stimmverhalten'),
    ],
    order=alt.Order('variable_order:O')
).transform_calculate(
    # Create an order field for proper stacking
    variable_order='if(datum.variable == "ja", 1, if(datum.variable == "enthaltung", 2, 3))',
    # Format numbers with thousand separators
    value_formatted='format(datum.value, ",.0f")',
    # Calculate and format percentage
    percentage_formatted='format(datum.value / datum.total_party, ".1%")'
).properties(
    width=600,
    height=400,
    title='Wahlverhalten der Wähler:innen nach Partei (geschätzt)'
)

chart

alt.Chart(...)

In [79]:
# Chart with absolute numbers (same sorting as percentage chart)
# Filter out "nichtwähler" for this chart
df_chart_no_nichtwähler = df_chart.filter(pl.col('partei') != 'nichtwahler')

chart_absolute = alt.Chart(df_chart_no_nichtwähler).mark_bar().encode(
    x=alt.X('sum(value):Q', 
            stack='zero', 
            title='Anzahl Stimmen'),
    y=alt.Y('partei:N', 
            title='Partei',
            sort=party_order),  # Same sorting as before (by ja percentage)
    color=alt.Color('variable:N', 
                   title='Stimmverhalten',
                   scale=alt.Scale(
                       domain=['ja', 'enthaltung', 'nein'],
                       range=['#1f77b4', '#7f7f7f', '#ff7f0e']  # blue, grey, orange
                   )),
    tooltip=[
        alt.Tooltip('partei', title='Partei'),
        alt.Tooltip('variable', title='Stimmverhalten'),
        alt.Tooltip('value:Q', title='Anzahl Stimmen', format=',.0f')
    ],
    order=alt.Order('variable_order:O')
).transform_calculate(
    # Create an order field for proper stacking
    variable_order='if(datum.variable == "ja", 1, if(datum.variable == "enthaltung", 2, 3))'
).properties(
    width=600,
    height=400,
    title='Wahlverhalten der Wähler:innen nach Partei (geschätzt) - Absolute Zahlen (ohne Nichtwähler)'
)

chart_absolute

alt.Chart(...)

In [69]:
farben = {
    'GRÜNE': '#84B547',
    'SP': '#F0554D',
    'PdA/Sol.': '#BF3939',
    'GLP': '#C4C43D',
    'Übrige': '#B8B8B8',
    'Lega': '#9070D4',
    'CSP': '#E3BA24',
    'EVP': '#DEAA28',
    'EDU': '#A65E42',
    'neuwähler': '#7F7F7F',
    'Mitte': '#D6862B',
    'SVP': '#4B8A3E',
    'FDP': '#3872B5',
    'SD': '#9D9D9D',
    'MCR': '#49A5E7',
    'FGA': '#A83232',
    'nichtwahler': '#D3D3D3',
}

In [70]:
# Create a chart with three stacked bars (ja, nein, enthaltung) showing parties in their colors
# Filter out nichtwähler for cleaner visualization
df_chart_parties = df_chart.filter(pl.col('partei') != 'nichtwahler')

# Filter party_order to exclude nichtwähler for consistent sorting
party_order_no_nichtwähler = [p for p in party_order if p != 'nichtwahler']

# Create interactive selection for parties (updated syntax)
party_selection = alt.selection_point(fields=['partei'], toggle=True)

# Main chart
chart_main = alt.Chart(df_chart_no_nichtwähler).add_params(
    party_selection
).mark_bar().encode(
    x=alt.X('variable:N', 
            title='Abstimmungsverhalten',
            sort=['ja', 'nein', 'enthaltung']),
    y=alt.Y('sum(value):Q', 
            stack='zero',
            title='Anzahl Stimmen'),
    color=alt.Color('partei:N', 
                   title='Partei',
                   sort=party_order,
                   scale=alt.Scale(
                       domain=list(farben.keys()),
                       range=list(farben.values())
                   ),
                   legend=None),  # Remove default legend
    opacity=alt.condition(party_selection, alt.value(1.0), alt.value(0.3)),
    tooltip=[
        alt.Tooltip('partei:N', title='Partei'),
        alt.Tooltip('variable:N', title='Stimmverhalten'),
        alt.Tooltip('value:Q', title='Anzahl Stimmen', format=',.0f')
    ]
).properties(
    width=400,
    height=800,
    title='Wahlverhalten nach Abstimmungstyp'
)

# Interactive legend
legend = alt.Chart(df_chart_parties).add_params(
    party_selection
).mark_rect(size=100).encode(
    y=alt.Y('partei:N',
            sort=party_order_no_nichtwähler,
            axis=alt.Axis(title='Partei', labelFontSize=11),
            scale=alt.Scale(paddingInner=0.1)),
    color=alt.Color('partei:N',
                   scale=alt.Scale(
                       domain=list(farben.keys()),
                       range=list(farben.values())
                   ),
                   legend=None),
    opacity=alt.condition(party_selection, alt.value(1.0), alt.value(0.3))
).properties(
    width=80,
    height=800,
    title='Klicken zum Auswählen'
)

# Combine chart and legend
chart_by_vote = alt.hconcat(chart_main, legend).resolve_scale(
    color='independent',
    y='shared'
)

chart_by_vote

alt.HConcatChart(...)

In [71]:
print(chart_by_vote.to_json())

{
  "$schema": "https://vega.github.io/schema/vega-lite/v5.20.1.json",
  "config": {
    "view": {
      "continuousHeight": 300,
      "continuousWidth": 300
    }
  },
  "datasets": {
    "data-e655591c7dfc17556f43612e26d60cc7": [
      {
        "partei": "FDP",
        "value": 9009,
        "variable": "ja"
      },
      {
        "partei": "SP",
        "value": 271396,
        "variable": "ja"
      },
      {
        "partei": "SVP",
        "value": 196,
        "variable": "ja"
      },
      {
        "partei": "EVP",
        "value": 429,
        "variable": "ja"
      },
      {
        "partei": "PdA/Sol.",
        "value": 12999,
        "variable": "ja"
      },
      {
        "partei": "FGA",
        "value": 4314,
        "variable": "ja"
      },
      {
        "partei": "GR\u00dcNE",
        "value": 161177,
        "variable": "ja"
      },
      {
        "partei": "SD",
        "value": 0,
        "variable": "ja"
      },
      {
        "partei": "EDU",
    